# XGBoost PM2.5 — full network, five-arm ablation, continuous surfaces

Runs on the **84-sensor network** (not the 18-sensor pilot) with the station split shared with the
CNN-LSTM branch (54 train / 13 calibration / 16 test), and renders the surfaces as a **continuous
gradient** rather than AQI tiers.

Works locally (from `XGBoost/`) and on Kaggle. Training is skipped automatically when
`outputs/ablation_results.csv` already holds all five arms.

Cell-by-cell guide and glossary: **[GUIDE.md](GUIDE.md)**.


In [ ]:
# ---- 0. environment: find the code, find the data, build whatever is missing -------------------
# Works with either Kaggle layout, so swapping the dataset no longer breaks the notebook:
#   (a) ONE dataset holding code + data                -> copied straight out
#   (b) a DATA-ONLY bundle + code from the public repo  -> data symlinked in, tables built here
# Locally every step is a no-op: the folder is already complete.
import glob, os, shutil, subprocess, sys

REPO = 'https://github.com/Patthethicc/urban-air.git'   # public; only used if no code dataset is attached
ROOT = '/kaggle/working/urban-air'
ON_KAGGLE = os.path.exists('/kaggle')
STATIONS = 'data/OpenAQ/locations.csv'                  # local fallback

# Set the switches BEFORE importing `data` anywhere: data.py reads them at import time to choose the
# master table and the fold file, and the module is cached afterwards.
os.environ['XGB_DATASET'] = 'full'      # the 84-sensor network (not the 18-sensor pilot)
os.environ['XGB_FOLDS'] = 'shared'      # the CNN-LSTM run's 54 / 13 / 16 station split


def _first(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None


if ON_KAGGLE:
    # 0. a zip upload may arrive unextracted -- Kaggle sometimes keeps an archive as one file.
    #    Unpack it once into a writable place; the search below then finds OpenAQ where it expects.
    if not glob.glob('/kaggle/input/**/OpenAQ', recursive=True):
        _zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
        if _zips:
            import zipfile
            _unpacked = '/kaggle/working/_bundle'
            os.makedirs(_unpacked, exist_ok=True)
            with zipfile.ZipFile(_zips[0]) as _z:
                _z.extractall(_unpacked)
            print('unpacked', _zips[0], '->', _unpacked)

    # 1. code: a dataset that holds the pipeline, else a clone of the repository.
    #    The repo keeps the pipeline one level down (urban-air/xgboost/), so CHDIR to wherever
    #    run_pipeline.py actually is. Everything below uses relative paths, so this is all it takes.
    hit = _first('/kaggle/input/**/run_pipeline.py')
    if hit:
        shutil.copytree(os.path.dirname(hit), ROOT, dirs_exist_ok=True)
        CODE = ROOT
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO, ROOT], check=True)
        inner = glob.glob(os.path.join(ROOT, '**', 'run_pipeline.py'), recursive=True)
        # a repository can hold more than one run_pipeline.py; prefer the folder that is actually the
        # pipeline (it must have data.py beside run_pipeline.py, and the preprocessing package).
        good = [os.path.dirname(p) for p in inner
                if os.path.exists(os.path.join(os.path.dirname(p), 'data.py'))]
        if not good:
            raise SystemExit('repo cloned, but no run_pipeline.py with data.py beside it was found')
        CODE = good[0]
    os.chdir(CODE)
    sys.path.insert(0, CODE)          # make `import data` unambiguous

    # Fail loudly if `data` resolved to something that is not the pipeline module: a stray `data/`
    # folder can be imported as an empty namespace package, which otherwise surfaces much later as
    # "module 'data' has no attribute 'ROOT'" with no clue where it came from.
    import data as _probe
    if not hasattr(_probe, 'ROOT'):
        raise SystemExit(
            'imported %r instead of the pipeline data.py (code folder: %s).\n'
            'Attach a dataset that contains the code -- run_pipeline.py AND data.py in the same folder.'
            % (getattr(_probe, '__file__', _probe), CODE))

    # 2. data: shipped beside the code, or a separate raw bundle (OpenAQ/, QC Data Converted/,
    #    Morphology/master_raster_v2.tif). Symlinks keep /kaggle/input read-only.
    if not (os.path.isdir('data') and os.listdir('data')):
        openaq = (_first('/kaggle/input/**/OpenAQ')
                  or _first('/kaggle/working/_bundle/**/OpenAQ'))   # the zip fallback unpacks here
        if not openaq:
            raise SystemExit('no data found: attach the dataset that contains OpenAQ/')
        bundle = os.path.dirname(openaq)
        print('data bundle:', bundle)
        os.makedirs('data', exist_ok=True)
        # link EVERYTHING the bundle provides: raw folders, the raster, any prebuilt table, and
        # .tile_cache. Whatever is present is used; whatever is missing gets built in step 3.
        BIG = 20 * 1024 * 1024
        for entry in sorted(os.listdir(bundle)):
            src_e, dst = os.path.join(bundle, entry), os.path.join('data', entry)
            if os.path.exists(dst):
                continue
            # A staged run rewrites fold_partitions/patch_stats/imputed_*/ingest_report, and writing
            # through a symlink into /kaggle/input fails with EROFS -- so small files are COPIED.
            # Directories and the two big read-only inputs (master 171 MB, raster 87 MB) are linked.
            if os.path.isdir(src_e) or os.path.getsize(src_e) > BIG:
                os.symlink(src_e, dst)
            else:
                shutil.copy2(src_e, dst)
        # The cache folder may be spelled differently in a hand-built bundle, and the cloned repo may
        # carry an older basemap.py that only knows one spelling. Normalise it to .tile_cache.
        for alt in ('.tilecache', 'tile_cache', 'tilecache'):
            src_alt = os.path.join('data', alt)
            dst_alt = os.path.join('data', '.tile_cache')
            if os.path.isdir(src_alt) and not os.path.exists(dst_alt):
                os.symlink(src_alt, dst_alt)
                print('tile cache linked: data/%s -> data/.tile_cache' % alt)
                break
        # the raster may sit in a Morphology/ subfolder of the original bundle
        tif = os.path.join('data', 'master_raster_v2.tif')
        if not os.path.exists(tif):
            for cand in (os.path.join(bundle, 'Morphology', 'master_raster_v2.tif'),
                         os.path.join(bundle, 'master_raster_v2.tif')):
                if os.path.exists(cand):
                    os.symlink(cand, tif)
                    break
        for cand in (os.path.join(bundle, 'Combined Dataset', 'full_with_preinstall', 'locations.csv'),
                     os.path.join(bundle, 'locations.csv')):
            if os.path.exists(cand):
                STATIONS = cand
                break

    # 3. derived tables: build only what is absent (a prebuilt dataset skips both steps)
    if not os.path.exists(os.path.join('data', 'Master_DL_Ready_full.csv')):
        print('building the 500 m morphology grid ...')
        subprocess.run([sys.executable, 'build_morphology_grid.py',
                        '--raster', 'data/master_raster_v2.tif',
                        '--openaq-locations', STATIONS,
                        '--qc-locations', 'data/QC Data Converted/QC Sensor Locations.csv'],
                       check=True)
        print('building Master_DL_Ready_full.csv (~4 min) ...')
        subprocess.run([sys.executable, 'ingest_sources.py'], check=True)

    # 4. the two packages Kaggle does not ship
    if subprocess.run([sys.executable, '-c', 'import skopt, shap'],
                      capture_output=True).returncode:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'scikit-optimize', 'shap'], check=True)

import data
print('root  :', data.ROOT)
print('folds :', os.path.basename(data.FOLD_JSON))
print('master:', os.path.basename(data.MASTER_CSV), '| exists:', os.path.exists(data.MASTER_CSV))
print('grid  :', os.path.basename(data.GRID_CSV), '| exists:', os.path.exists(data.GRID_CSV))
assert os.path.exists(data.MASTER_CSV), 'data table missing -- read the notes above'
assert os.path.exists(data.GRID_CSV), 'grid missing -- build_morphology_grid.py did not run'


## 1. Data and features

`preprocessing.prep` runs on import: it merges the 500 m morphology grid onto the sensor table, builds
gap-aware hourly lags (`pm25_lag_1..4`, 24 h and 72 h rolling means/stds) and the cyclical time terms.
The 10 m raster is the *source* of the morphology; the model never sees a 10 m pixel.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from preprocessing import prep
from preprocessing import spatial_cv
from training.ablation import ARM_FEATURES, ARM_LABELS

df_clean = prep.df_clean
MAPPING = data.SENSOR_MAP_CSV
print('rows %s | cells %d | stations %d | features %d'
      % (f'{len(df_clean):,}', df_clean['Grid_ID'].nunique(),
         df_clean['Location'].nunique(), len(prep.features)))
print('period %s .. %s' % (df_clean['DateTime'].min().date(), df_clean['DateTime'].max().date()))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))

axes[0].hist(df_clean['Value'], bins=60, color='#2a6f77', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_title('PM2.5 distribution (raw µg/m³)')
axes[0].set_xlabel('PM2.5 (µg/m³)'); axes[0].set_ylabel('rows (log)')

per_cell = df_clean.groupby('Grid_ID').size().sort_values()
axes[1].barh(per_cell.index.astype(str), per_cell.values, color='#c1666b')
axes[1].set_title('Rows per grid cell'); axes[1].set_xlabel('rows')
axes[1].tick_params(labelsize=4)

by_src = df_clean.groupby('source').size() if 'source' in df_clean.columns else None
if by_src is not None:
    axes[2].bar(by_src.index.astype(str), by_src.values, color='#7f8c8d')
    axes[2].set_title('Rows by network'); axes[2].set_ylabel('rows')
plt.tight_layout(); plt.show()

## 1b. EDA findings and the cleaning policy

`eda/eda_results/EDA_Report.md` audits the raw network and makes four recommendations. What each one
means for THIS model, measured rather than assumed:

| EDA finding | Rows affected | Applied here | Measured effect |
|---|---|---|---|
| **Sensor-floor readings** (`<= 0.5 ug/m3`) | 5,292 (0.36%) | **dropped**, before the lags | keeps a 0.0 from becoming next hour's `pm25_lag_1`; model change ~0.3% |
| **New Year's Eve fireworks** (300-529 ug/m3, Dec 31 21:00 - Jan 1 06:00) | 1,389 (0.09%) | **opt-in** (`XGB_CLEAN_NYE=on`) | training on masked data changes the model by ~0.3%, but the *reported* RMSE falls 1.9% and **R2 falls 0.010** purely because 363 extreme rows leave the test set |
| **Staggered QC deployment cohorts** (8 -> 20 -> 40 sensors, 2021-2024) | -- | **not applied as a filter** | the lag now requires >= 2 contributing stations instead of thinning the years |
| **Raw wide-format quirks** (`QC Data/`) | -- | not applicable | this pipeline reads the long-format `QC Data Converted/`, not the wide files |

Two conclusions worth keeping:

1. **NYE masking is deliberately OFF by default.** It lowers RMSE by deleting the hardest rows, which is
   metric cherry-picking unless reported; the model itself is unchanged. `--XGB_CLEAN_NYE=on` reproduces
   it as a sensitivity row.
2. **956.8 ug/m3 occurs outside the fireworks window**, so the tail is not purely an artefact and was not
   winsorised globally. The tail is real; the model's failure on it is the finding, not noise to remove.

See `experiments/eda_cleaning_impact.py` for the paired comparison that produced those numbers.


In [ ]:
# what the cleaning did, and how much of the squared error sits in the tail
# NOTE: this cell runs BEFORE the ablation (cell ARM_FILE/BEST are defined later), so it must not depend
# on them -- an earlier version did, and raised NameError on Kaggle's first run.
import os
import numpy as np
import pandas as pd

print('cleaning policy: floor <= 0.5 ug/m3 -> %s | NYE masking -> %s'
      % (os.environ.get('XGB_CLEAN_FLOOR', 'on'), os.environ.get('XGB_CLEAN_NYE', 'off')))
print('rows in df_clean: %s' % f'{len(df_clean):,}')

# the outlier structure is a property of the TARGET, so it is available before any model is fitted
v = df_clean['Value']
for thr in (100, 150):
    print('  %.3f%% of rows have PM2.5 >= %d ug/m3   (max %.1f)'
          % (100.0 * (v >= thr).mean(), thr, v.max()))

# the share of SQUARED ERROR needs out-of-fold predictions, so only compute it if a previous run left
# them; on a clean Kaggle run this cell must pass without them
ARM_FILE_PREVIEW = {'A': 'A', 'B': 'B', 'C': 'C', 'C+': 'Cplus', 'TEXTURE': 'TEXTURE'}
_arm_now = BEST if 'BEST' in dir() else 'A'
_oof_path = os.path.join('outputs', 'oof_predictions_%s.csv' % ARM_FILE_PREVIEW[_arm_now])
if os.path.exists(_oof_path):
    _o = pd.read_csv(_oof_path, usecols=['actual', 'predicted'])
    _e2 = (_o.actual - _o.predicted) ** 2
    print('  tail share: %.3f%% of rows have actual > 150 and carry %.1f%% of the squared error'
          % (100.0 * (_o.actual > 150).mean(), 100.0 * _e2[_o.actual > 150].sum() / _e2.sum()))
else:
    print('  tail share of squared error: reported in the results section, once the ablation has run')


## 2. Dataset report (Deliverable 1)

In [ ]:
!python run_pipeline.py --stage report
from IPython.display import Markdown, display
display(Markdown(open('outputs/dataset_report.md', encoding='utf-8').read()[:3000]))

## 3. Spatial folds

Cells are clustered by centroid (k-means) into spatially contiguous, non-overlapping folds, so no cell
used for training is in the held-out fold — the thesis requirement (Fig. 4.16).

In [ ]:
!python run_pipeline.py --stage split
!python run_pipeline.py --stage patch
doc = spatial_cv.load_fold_map()
parts = doc['partitions']
print('train %d / calibration %d / test %d cells'
      % (len(parts['train']), len(parts['calibration']), len(parts['test'])))

In [ ]:
cent = spatial_cv.get_grid_centroids(MAPPING).set_index('Grid_ID')
outer_of = {int(c): int(f) for f, cs in doc['folds']['outer'].items() for c in cs}
palette = ['#2a6f77', '#c1666b', '#e0a458']

fig, ax = plt.subplots(figsize=(6.4, 6.0))
for name, col, mk in [('train', None, 'o'), ('calibration', '#8e44ad', 's'), ('test', '#111111', '*')]:
    cs = parts[name]
    pts = cent.loc[[c for c in cs if c in cent.index]]
    if name == 'train':
        for ci, cc in enumerate(pts.index):
            ax.scatter(pts.loc[cc, 'cx'], pts.loc[cc, 'cy'], s=26,
                       color=palette[outer_of.get(int(cc), 0) % 3], marker='o', alpha=0.85)
    else:
        ax.scatter(pts['cx'], pts['cy'], s=110 if name == 'test' else 60, color=col,
                   marker=mk, edgecolor='white', linewidth=0.6, label=name, zorder=5)
ax.set_aspect('equal'); ax.set_xlabel('easting (m)'); ax.set_ylabel('northing (m)')
ax.set_title('Spatial partitions\n(train coloured by outer fold; test = stars)', fontsize=10, loc='left')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 4. Five-arm ablation

`A` time only · `B` morphology+time · `C` full · `C+` full+spatial lag · `TEXTURE` full+10 m patch
statistics. Nested spatial CV: 3 outer folds, 3 inner folds, Bayesian (GP-EI) hyperparameter search.

In [ ]:
ARMS = ['A', 'B', 'C', 'C+', 'TEXTURE']
_p = 'outputs/ablation_results.csv'
HAVE_RESULTS = False
if os.path.exists(_p):
    _r = pd.read_csv(_p)
    _per = _r.groupby('arm').size()
    HAVE_RESULTS = set(ARMS) <= set(_r['arm']) and all(_per.get(a, 0) >= 3 for a in ARMS)
    print('existing folds per arm:', {a: int(_per.get(a, 0)) for a in ARMS})
print('reuse existing results:', HAVE_RESULTS)

In [ ]:
# 60 trials on every 4th training row costs less than the original 20 trials on every row, and the
# search actually converges. Validation rows are never strided, so early stopping is unaffected.
N_TRIALS = 60
ROW_STRIDE = 4

if HAVE_RESULTS:
    print('skipping training: all 5 arms x 3 folds already in outputs/ablation_results.csv')
else:
    !python run_pipeline.py --stage ablation --arms A B C C+ TEXTURE --n-trials {N_TRIALS} --row-stride {ROW_STRIDE}

## 4b. The untuned control

`STATUS.md` item 5 flags that the arms differ in tuning as well as features, and says a fixed-parameter
run is the tighter control. This is that run: arm C, XGBoost's own defaults, no search at all.

It answers one question — **what did 20 Bayesian trials buy?** — and it writes to `outputs/untuned/`, so
the tuned results in `outputs/` are untouched.


In [ ]:
# ---- untuned control: same folds, default parameters, no search --------------------------------
import pandas as pd

UNTUNED_DIR = 'outputs/untuned'
!python run_pipeline.py --stage ablation --arms C --untuned --ablation-out-dir {UNTUNED_DIR}

tuned = pd.read_csv('outputs/ablation_summary.csv').set_index('arm')
unt = pd.read_csv(UNTUNED_DIR + '/ablation_summary.csv').set_index('arm')
cmp = pd.DataFrame({
    'tuned pooled R2': tuned.loc['C', ['pooled_r2']],
    'untuned pooled R2': unt.loc['C', ['pooled_r2']],
}).T
cmp['pooled_rmse'] = [tuned.loc['C', 'pooled_rmse'], unt.loc['C', 'pooled_rmse']]
cmp['pooled_mae'] = [tuned.loc['C', 'pooled_mae'], unt.loc['C', 'pooled_mae']]
print(cmp.round(4).to_string())
gain = float(tuned.loc['C', 'pooled_r2'] - unt.loc['C', 'pooled_r2'])
print('\nsearch gain on arm C: %+.4f pooled R2  (tuned - untuned)' % gain)
if gain > 0.005:
    print('  the search earned its trials')
elif gain < -0.005:
    print('  the search HURT: report this honestly -- the tuned arm is worse than the defaults')
else:
    print('  the search bought nothing measurable on this arm')


In [ ]:
# the ablation writes this; without it there is no BEST arm to score
if not os.path.exists('outputs/ablation_summary.csv'):
    raise SystemExit('outputs/ablation_summary.csv missing -- run the ablation cell above '
                     '(five arms, ~2.5 h) before this one')
# arm names are not safe filenames ('C+' -> 'Cplus'), and BEST is used by later cells
summary = pd.read_csv('outputs/ablation_summary.csv').sort_values('pooled_r2', ascending=False)
BEST = summary['arm'].iloc[0]
ARM_FILE = {'A': 'A', 'B': 'B', 'C': 'C', 'C+': 'Cplus', 'TEXTURE': 'TEXTURE'}
print(summary[['arm','label','n_features','pooled_rmse','pooled_mae','pooled_r2',
               'r2_mean','r2_sd','oof_rows']].round(3).to_string(index=False))
print('\nwinning arm:', BEST)


In [ ]:
shown = list(dict.fromkeys(['A', 'C', BEST]))      # dedupe if BEST is A or C
oof = {a: pd.read_csv('outputs/oof_predictions_%s.csv' % ARM_FILE[a]) for a in shown}
lims = [0, float(np.nanpercentile(oof[shown[0]]['actual'], 99.5))]

fig, axes = plt.subplots(1, len(shown), figsize=(4.4 * len(shown), 3.9), sharex=True, sharey=True,
                         squeeze=False)
for ax, a in zip(axes.ravel(), shown):
    d = oof[a]
    ax.scatter(d['actual'], d['predicted'], s=5, alpha=0.25, color='#2a6f77', edgecolors='none')
    ax.plot(lims, lims, 'k--', lw=1)
    r2 = summary.set_index('arm').loc[a, 'pooled_r2']
    ax.set_title('%s  (R2 %.3f)' % (ARM_LABELS[a], r2), fontsize=9)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('actual PM2.5')
axes.ravel()[0].set_ylabel('predicted PM2.5')
plt.tight_layout(); plt.show()


## 5. The reserved test stations

Never used for training, tuning or calibration. This is the headline accuracy.

In [ ]:
!python run_pipeline.py --stage test --test-arm {BEST} --test-seeds 42 43 44
print(pd.read_csv('outputs/test_metrics_all.csv')[['arm','stations','rmse','mae','r2','ia',
                                                   'pearson_r','bias','within_5','within_10']]
      .round(3).to_string(index=False))

## 6. Spatial bias in the out-of-fold residuals (Section 4.4.3)

If errors cluster geographically, a purely aspatial model is missing something the map could supply.
Moran's I on the per-cell mean residual answers that directly.

In [ ]:
from preprocessing.spatial_bias import (aggregate_residuals_by_grid, build_knn_weights,
                                        global_morans_i, local_morans_i)

oof_df = pd.read_csv(f'outputs/oof_predictions_{ARM_FILE[BEST]}.csv')
grid_res = aggregate_residuals_by_grid(oof_df, MAPPING)
print('aggregated to %d grid cells' % len(grid_res))

w = build_knn_weights(grid_res, k=4)
I, p_value = global_morans_i(grid_res['residual'].values, w)
print("Global Moran's I: %+.3f   (p = %.3f, %d cells)" % (I, p_value, len(grid_res)))

if p_value < 0.05:
    lisa = pd.concat([grid_res[['Grid_ID','residual']].reset_index(drop=True),
                      local_morans_i(grid_res['residual'].values, w)], axis=1)
    print(lisa.sort_values('p_sim').head(10).to_string(index=False))
else:
    print('-> no statistically significant spatial clustering of residuals at this alpha')

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.8))
sizes = 80 + 2600 * (grid_res['residual'].abs() / grid_res['residual'].abs().max())
colors = np.where(grid_res['residual'] >= 0, '#c1666b', '#2a6f77')
ax.scatter(grid_res['cx'], grid_res['cy'], s=sizes, c=colors, alpha=0.75, edgecolor='white',
           linewidth=0.5)
ax.set_aspect('equal')
ax.set_title('Mean out-of-fold residual per cell\n(red = over-prediction, teal = under-prediction)',
             fontsize=10, loc='left')
ax.set_xlabel('easting (m)'); ax.set_ylabel('northing (m)')
plt.tight_layout(); plt.show()

## 7. Prediction intervals — GeoCP, matched to the CNN-LSTM settings

Kernel × bandwidth grid `[2000, 5000, 10000, 20000, inf]` with gaussian and exponential kernels, selected
by leave-one-station-out on the calibration stations, `finite_sample_correction = False`.

In [ ]:
!python run_pipeline.py --stage geocp --geocp-profile shared --max-cal 6000 --max-eval 20000

In [ ]:
import json
g = json.load(open('outputs/geocp_shared/geocp.json'))
print('selected kernel :', g['selected']['kernel'], '@', g['selected']['bandwidth_m'], 'm')
print('LOSO coverage   : %.3f' % g['selected']['loso_coverage'])
print('TEST coverage   : %.4f  (target %.2f)' % (g['test']['coverage'], g['test']['target_coverage']))
print('mean width      : %.2f ug/m3' % g['test']['mean_width'])
cand = pd.read_csv('outputs/geocp_shared/geocp_candidates.csv')
display(cand.round(3).sort_values('loso_coverage', ascending=False).head(8))


## 8. The model artifact

In [ ]:
!python run_pipeline.py --stage save
print(open('outputs/artifacts/latest/features.json', encoding='utf-8').read()[:400])

## 9. Interpretability — gain importance and TreeSHAP

Grad-CAM does not apply to a tree ensemble (there is no convolutional feature map to attribute), so
attribution uses TreeSHAP, which is exact and additive for tree models.

In [ ]:
import joblib
model = joblib.load('outputs/artifacts/latest/model.joblib')
gain = pd.Series(model.get_booster().get_score(importance_type='gain'))
gain.index = [int(i[1:]) for i in gain.index]          # 'f0' -> 0
gain = gain.reindex(range(len(ARM_FEATURES[BEST]))).fillna(0)
gain.index = ARM_FEATURES[BEST]
gain = (gain / gain.sum()).sort_values()
print(gain.tail(10).round(3).to_string())


In [ ]:
morph_cols = ['bh_mean','bh_stdev','ndvi_mean','bldg_dens','aspect_ratio_mean','street_width_mean']
present = [c for c in morph_cols if c in gain.index]
morph_share = 100.0 * gain[present].sum() if present else 0.0
print('gain share of the 6 morphological features: %.1f%%' % morph_share)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
colors = ['#c0392b' if f in morph_cols else '#2a6f77' for f in gain.index]
ax.barh(range(len(gain)), gain.values, color=colors)
ax.set_yticks(range(len(gain))); ax.set_yticklabels(gain.index, fontsize=7)
ax.set_xlabel('share of total gain')
ax.set_title('Gain importance, arm %s (morphology in red)' % BEST, fontsize=10, loc='left')
plt.tight_layout(); plt.show()


In [ ]:
!python run_pipeline.py --stage figures --figures-arm {BEST}

In [ ]:
from IPython.display import Image, display
for f in ['shap_beeswarm.png','shap_waterfall.png','shap_importance.png','pred_vs_obs.png',
          'error_vs_concentration.png','accuracy_by_station.png','residual_hist.png',
          'timeseries_sample.png','geocp_coverage.png']:
    p = os.path.join('outputs/figures', f)
    if os.path.exists(p):
        print(f); display(Image(p, width=640))

## 10. City-wide surfaces — continuous gradient

Four snapshots on one shared colour scale, plus a 24-hour mean. No AQI tiers: this pipeline's
predictions cluster in a narrow band, which put almost every cell in the same AQI bin and made the tiered
map a flat colour.

In [ ]:
# Two days for comparison: a clean one and a polluted one.
# The tile cache is keyed by extent+zoom, not by date, so both render with the same OSM background.
NORMAL_DAY   = '2025-07-23'    # network mean 11.8 ug/m3, max  91.9
POLLUTED_DAY = '2025-08-30'    # network mean 33.0 ug/m3, max 173.0
DATES = [NORMAL_DAY, POLLUTED_DAY]

for d in DATES:
    print('=' * 74)
    print('rendering', d)
    !python run_pipeline.py --stage infer --date {d}


In [ ]:
from IPython.display import Image, display
import os

for d in DATES:
    day = d.replace('-', '')
    out = os.path.join('outputs', 'surfaces', day)
    print('=' * 74)
    print(d, '   (same colour rule as its own panel; the two days differ in scale)')
    for name in (f'panel_{day}.png', f'{day}_mean_continuous.png'):
        p = os.path.join(out, name)
        if os.path.exists(p):
            print(' ', name)
            display(Image(p, width=780))
        else:
            print('  MISSING:', p)


## 12. Window sweep — does the null survive the date range?

Off by default, because four windows × five arms is roughly **7.5 h**, and a Kaggle CPU session stops at
12 h. Run it as its own commit (or locally overnight) when you have the time:

```python
RUN_WINDOW_SWEEP = True      # in the cell below
```

Or from a shell, which also gives you a dry run first:

```bash
python experiments/window_sweep.py --dry-run
python experiments/window_sweep.py --arms A C C+ --n-trials 20    # ~1 h, three arms
```

Each window runs in its own subprocess with `XGB_WINDOW_START` set and writes to `outputs/sweep/<date>/`,
so nothing here can overwrite the main results. What you are looking for is whether **C − A** stays near
zero in every window; if one window flips it positive by more than ~0.02, that window needs a paragraph.


In [ ]:
# ---- cohort/window sweep: OPT-IN -----------------------------------------------------------------
RUN_WINDOW_SWEEP = False        # four windows x five arms is ~7.5 h; keep it off for the main commit

if not RUN_WINDOW_SWEEP:
    print('window sweep skipped (RUN_WINDOW_SWEEP = False).')
    print('run it with: python experiments/window_sweep.py --dry-run   then drop the --dry-run')
else:
    import pandas as pd
    !python experiments/window_sweep.py --arms A B C C+ TEXTURE --n-trials 60 --row-stride 4
    sweep = pd.read_csv('outputs/sweep/window_sweep.csv')
    print(sweep.pivot_table(index='arm', columns='window', values='pooled_r2').round(3).to_string())
    p = sweep.pivot_table(index='arm', columns='window', values='pooled_r2')
    if {'C', 'A'} <= set(p.index):
        print('\nC - A by window (near zero = the null survives):')
        print((p.loc['C'] - p.loc['A']).round(3).to_string())


## 10b. Verification — the headline numbers, re-derived

Two things happen here, and both are cheap:

1. Every arm is scored on the 16 reserved test stations (each fit takes ~10 s at deployment size), which
   is what makes `arm_comparison.png` possible.
2. The four property checks in `checks/` are run. They read what this run wrote and recompute the
   numbers from scratch, so a pass here means the tables below were derived, not copied.

`--stage checks` is deliberately **not** used: it also runs `run_notebook.py`, which would execute this
whole notebook a second time. Of the individual scripts, only the two fastest are called here — `verify_geocp.py` (>150 s) and `verify_shared_split.py` (>25 min) repeat the whole LOSO search and the
raster patch statistics respectively, so run those by hand.


In [ ]:
# ---- re-derive the headline numbers from what this run wrote -----------------------------------
import glob, subprocess, sys

for _a in ('A', 'B', 'C', 'C+', 'TEXTURE'):
    subprocess.run([sys.executable, 'run_pipeline.py', '--stage', 'test', '--test-arm', _a], check=True)

# root-level figures (arm_comparison + the shap json the summary reads)
subprocess.run([sys.executable, 'run_pipeline.py', '--stage', 'figures'], check=True)

import pandas as pd
abl = pd.read_csv('outputs/ablation_summary.csv')
test = pd.read_csv('outputs/test_metrics_all.csv')

print('=== out-of-fold, training cells only (nested spatial CV) ===')
print(abl.sort_values('pooled_r2', ascending=False)
      [['arm', 'label', 'n_features', 'pooled_r2', 'r2_mean', 'r2_sd', 'pooled_rmse', 'pooled_mae']]
      .round(3).to_string(index=False))

print('\n=== the 16 reserved test stations (never fitted, tuned or calibrated on) ===')
cols = ['arm', 'stations', 'r2'] + [c for c in ('r2_seed_sd',) if c in test.columns] + \
       ['rmse', 'mae', 'bias', 'within_5', 'within_10']
print(test.sort_values('r2', ascending=False)[cols].round(3).to_string(index=False))
if 'r2_seed_sd' in test.columns:
    print('\nr2_seed_sd is the spread of the per-seed R2 -- the ensemble is the reported r2')

print('\n=== agreement between the two views ===')
best_t = test.sort_values('r2', ascending=False)['arm'].iloc[0]
best_o = abl.sort_values('pooled_r2', ascending=False)['arm'].iloc[0]
print('best on test stations : %s' % best_t)
print('best out-of-fold      : %s' % best_o)
print('same arm             : %s' % (best_t == best_o))

# Only the checks that finish in seconds. Timed on this machine:
#   verify_ablation.py          50 s   <- recomputes R2 from the out-of-fold predictions
#   check_morphology_coverage.py 7 s
#   verify_geocp.py            >150 s  (repeats the whole LOSO search)
#   verify_shared_split.py     >25 min (recomputes the raster patch statistics)
# Run the slow two by hand when you want them; they are not worth stalling a commit for.
FAILED = []
for _s in ('verify_ablation.py', 'check_morphology_coverage.py'):
    try:
        _r = subprocess.run([sys.executable, 'checks/' + _s], capture_output=True, text=True,
                            timeout=600)
        ok, _out = _r.returncode == 0, (_r.stdout or _r.stderr)
    except subprocess.TimeoutExpired:
        ok, _out = False, 'timed out after 600 s'
    print('\n--- %-32s %s' % (_s, 'ok' if ok else 'FAILED'))
    print(_out.strip()[-800:])
    if not ok:
        FAILED.append(_s)
print('\nVERIFICATION:', 'all checks passed' if not FAILED else 'FAILED -> %s' % FAILED)


## 11. Summary

In [ ]:
import glob
print('network      :', df_clean['Grid_ID'].nunique(), 'cells /', df_clean['Location'].nunique(),
      'stations,', f'{len(df_clean):,}', 'rows')
print('winning arm  :', BEST, '(%d features)' % len(ARM_FEATURES[BEST]))
t = pd.read_csv('outputs/test_metrics_all.csv').set_index('arm').loc[BEST]
print('test stations: R² %.3f | MAE %.3f | RMSE %.3f | bias %+.3f'
      % (t['r2'], t['mae'], t['rmse'], t['bias']))
print('intervals    : coverage %.3f (target %.2f), width %.1f µg/m³'
      % (g['test']['coverage'], g['test']['target_coverage'], g['test']['mean_width']))
# --figures-arm writes into outputs/figures/<arm>/, so look there too instead of assuming the root
_shap_hits = sorted(glob.glob('outputs/figures/**/shap_importance.json', recursive=True),
                    key=lambda p: (BEST not in p, p))
_morph_shap = (json.load(open(_shap_hits[0], encoding='utf-8'))['families_pct'].get('morphology', 0.0)
               if _shap_hits else float('nan'))
print('morphology   : %.1f%% of gain, %.1f%% of SHAP attribution  (%s)'
      % (morph_share, _morph_shap, _shap_hits[0] if _shap_hits else 'no shap json found'))